## Final Heart Rate Code !

We will only keep the the last heart rate measurement before the survey time plus a column for the log(diff_sec).



In [ ]:
import os
import sys

import pandas as pd

sys.path.append(os.path.abspath(".."))  # repo root, for anchor.py
from anchor import data_dir

PERSONAI_PATH = os.path.join(data_dir(), "personai")

# folder with the per-user heart-rate files "<user>_hr_aggregated.csv" (not part of the repo, set it to where they are)
HR_FOLDER = os.path.join(PERSONAI_PATH, "HR_Aggregated")

# intermediate file
NO_HR_COLS = os.path.join(PERSONAI_PATH, "all_aircontrol_with_surveys_hr_FIXED_no_hr_cols.csv")

# Load your CSV
df = pd.read_csv(os.path.join(PERSONAI_PATH, "all_aircontrol_with_surveys_hr_FIXED.csv"))

# Drop the unwanted columns
cols_to_drop = ["survey_time_eff", "bpm", "bpm_method", "deviation"]
df = df.drop(columns=cols_to_drop)

# Save to a new CSV (next to this notebook)
df.to_csv(NO_HR_COLS, index=False)


In [ ]:
import pandas as pd
import numpy as np
import os

# -----------------------------
# 1. Load main cleaned CSV
# -----------------------------
main_path = NO_HR_COLS
df = pd.read_csv(main_path)

df["survey_time"] = pd.to_datetime(df["survey_time"])

# -------------------------------------------------
# 2. Prepare HR folder and new columns
# -------------------------------------------------
hr_folder = HR_FOLDER
if not os.path.isdir(hr_folder):
    raise FileNotFoundError(f"Heart-rate folder not found: {hr_folder}. Set HR_FOLDER in the first cell.")

df["hr_time"] = pd.NaT
df["hr_bpm"] = np.nan
df["hr_source"] = np.nan
df["hr_time_diff_seconds"] = np.nan
df["hr_time_diff_log"] = np.nan

# Fix dtypes to avoid warnings
df["hr_time"] = pd.to_datetime(df["hr_time"])
df["hr_bpm"] = pd.to_numeric(df["hr_bpm"])
df["hr_source"] = df["hr_source"].astype("object")

# -------------------------------------------------
# 3. Process user-by-user
# -------------------------------------------------
unique_users = df["user_id"].unique()

for user in unique_users:

    print(f"Processing user: {user}")

    hr_file = os.path.join(hr_folder, f"{user}_hr_aggregated.csv")

    if not os.path.exists(hr_file):
        print(f"⚠️ HR file missing for user {user}, skipping.")
        continue

    hr = pd.read_csv(hr_file)
    hr["time"] = pd.to_datetime(hr["time"], format="ISO8601")

    mask = df["user_id"] == user
    df_user = df.loc[mask]

    for idx, row in df_user.iterrows():

        survey_t = row["survey_time"]

        hr_before = hr[hr["time"] <= survey_t]

        if hr_before.empty:
            continue

        hr_last = hr_before.iloc[-1]

        df.at[idx, "hr_time"] = hr_last["time"]
        df.at[idx, "hr_bpm"] = hr_last["bpm"]
        df.at[idx, "hr_source"] = hr_last["source"]
        df.at[idx, "deviation"] = hr_last["deviation"]

        diff_sec = (survey_t - hr_last["time"]).total_seconds()
        df.at[idx, "hr_time_diff_seconds"] = diff_sec
        if diff_sec > 0:
            df.at[idx, "hr_time_diff_log"] = np.log(diff_sec)
        elif diff_sec == 0:
            df.at[idx, "hr_time_diff_log"] = 0.0   
        else:
            df.at[idx, "hr_time_diff_log"] = np.nan


    print(f"✔ Finished user: {user}")

# -------------------------------------------------
# 4. Save result
# -------------------------------------------------
output_path = os.path.join(PERSONAI_PATH, "all_aircontrol_with_surveys_hr_with_new_bpm.csv")
df.to_csv(output_path, index=False)

print("=============================================")
print("🎉 Done! Final merged file saved as:")
print(output_path)
print("=============================================")


In [ ]:
# Overall max (ignoring NaNs)
max_diff = df["hr_time_diff_seconds"].max()
print("Max time diff (seconds):", max_diff)
mean = df["hr_time_diff_seconds"].mean()
std = df["hr_time_diff_seconds"].std()
print("Mean of the time diff(seconds):",mean)
print("STD of the time diff(seconds):",std)

In [ ]:
import matplotlib.pyplot as plt
df_plot_both = df.dropna(subset=["hr_time_diff_seconds", "hr_time_diff_log"])

plt.figure(figsize=(6, 6))
plt.scatter(df_plot_both["hr_time_diff_seconds"], df_plot_both["hr_time_diff_log"], s=5)
plt.xlabel("hr_time_diff_seconds")
plt.ylabel("hr_time_diff_log")
plt.title("Raw vs log HR–Survey time difference")
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np

df["z_seconds"] = (df["hr_time_diff_seconds"] - df["hr_time_diff_seconds"].mean()) / df["hr_time_diff_seconds"].std()

# Outliers: z-score > 3 or < -3
outliers_z = df[np.abs(df["z_seconds"]) > 3]

print("Number of outliers (z-score):", len(outliers_z))
outliers_z.head()


## Renaming Columns

In [ ]:
import pandas as pd

# Read the CSV
df = pd.read_csv(os.path.join(PERSONAI_PATH, "all_aircontrol_with_surveys_hr_with_new_bpm.csv"))

# Rename columns
df = df.rename(columns={
    "hr_time": "survey_time_eff",
    "hr_bpm": "bpm",
    "hr_time_diff_log": "bpm_time_diff_log"
})
df = df.drop(columns=["hr_source"])
df = df.drop(columns=["hr_time_diff_seconds"])
# Save back to CSV (overwrite or use a new file name)
df.to_csv(os.path.join(PERSONAI_PATH, "all_aircontrol_with_surveys_hr_with_new_bpm_renamed.csv"), index=False)
